## NSFG pregnancy data 

Key ideas in this notebook:
- **caseid** identifies a woman, while **pregordr** tells us the order of the pregnancy for that woman.
- **outcome** records whether the pregnancy ended in a live birth, miscarriage, abortion, stillbirth, or another result.
- **birthwgt_lb** and **birthwgt_oz** are birth-weight variables; the survey sometimes uses special codes for missing or not-applicable values, which we treat as missing data.
- **agepreg** is the mother's age at pregnancy, stored in a way that needs to be converted to a more natural scale before analysis.
- **prglngth** is the length of the pregnancy in weeks, and **birthord** records the birth order of the child.

These steps are typical in exploratory data analysis: inspect the distribution of values, clean the data, and then compute meaningful summaries such as means and frequencies.

In [65]:


# without manually copying them into the workspace.
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")


In [66]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [67]:
%pip install statsmodels

You should consider upgrading via the '/Users/ayushthasale07/Sarg<3/EDA/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [68]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML
from thinkstats import decorate



In [69]:


try:
    import statadict
except ImportError:
    %pip install statadict



In [70]:


dct_file = "./data/2002FemPreg.dct"
dat_file = "./data/2002FemPreg.dat.gz"



In [71]:
# this function reads the NSFG data file using a Stata dictionary.
# The original data are stored in a fixed-width text format, so the .dct file tells us
# where each variable begins and ends. This is a common structure in survey data.
# We use the dictionary to attach meaningful column names to the raw data.
from statadict import parse_stata_dict


def read_stata(dct_file, dat_file):
    stata_dict = parse_stata_dict(dct_file)
    resp = pd.read_fwf(
        dat_file,
        names=stata_dict.names,
        colspecs=stata_dict.colspecs,
        compression="gzip",
    )
    return resp


In [72]:
preg = read_stata(dct_file, dat_file)

In [73]:
preg.shape

(13593, 243)

In [74]:
preg.head()

,caseid,pregordr,howpreg_n,howpreg_p,moscurrp,nowprgdk,pregend1,pregend2,nbrnaliv,multbrth,...,poverty_i,laborfor_i,religion_i,metro_i,basewgt,adj_mod_basewgt,finalwgt,secu_p,sest,cmintvw
0,1,1,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
1,1,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
2,2,1,NaN,NaN,NaN,NaN,5.0,NaN,3.0,5.0,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
3,2,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
4,2,3,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231


In [75]:
preg.columns

Index(['caseid', 'pregordr', 'howpreg_n', 'howpreg_p', 'moscurrp', 'nowprgdk',
       'pregend1', 'pregend2', 'nbrnaliv', 'multbrth',
       ...
       'poverty_i', 'laborfor_i', 'religion_i', 'metro_i', 'basewgt',
       'adj_mod_basewgt', 'finalwgt', 'secu_p', 'sest', 'cmintvw'],
      dtype='object', length=243)

In [76]:

def show_table(d):
    df = pd.DataFrame(d)
    return HTML(df.to_html(index=False))

In [77]:

d = {
    "Value": [1, 2, 3, 4, 5, 6, "Total"],
    "Label": [
        "LIVE BIRTH",
        "INDUCED ABORTION",
        "STILLBIRTH",
        "MISCARRIAGE",
        "ECTOPIC PREGNANCY",
        "CURRENT PREGNANCY",
        "",
    ],
    "Total": [9148, 1862, 120, 1921, 190, 352, 13593],
}

show_table(d)

Value,Label,Total
1,LIVE BIRTH,9148
2,INDUCED ABORTION,1862
3,STILLBIRTH,120
4,MISCARRIAGE,1921
5,ECTOPIC PREGNANCY,190
6,CURRENT PREGNANCY,352
Total,,13593


In [78]:
# birthwgt_lb stores birth weight in pounds.
# Before analyzing it, we inspect its distribution with value_counts().
# This is a good example of a descriptive step from ThinkStats: look at the data shape
# before deciding how to clean or summarize it.
preg["outcome"].value_counts().sort_index()

outcome
1    9148
2    1862
3     120
4    1921
5     190
6     352
Name: count, dtype: int64

In [79]:

counts = preg["birthwgt_lb"].value_counts(dropna=False).sort_index()
counts

birthwgt_lb
0.0        8
1.0       40
2.0       53
3.0       98
4.0      229
5.0      697
6.0     2223
7.0     3049
8.0     1889
9.0      623
10.0     132
11.0      26
12.0      10
13.0       3
14.0       3
15.0       1
51.0       1
97.0       1
98.0       1
99.0      57
NaN     4449
Name: count, dtype: int64

In [80]:
counts.loc[0:5]

birthwgt_lb
0.0      8
1.0     40
2.0     53
3.0     98
4.0    229
5.0    697
Name: count, dtype: int64

In [81]:

preg["birthwgt_lb"] = preg["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)

In [82]:

preg["agepreg"].mean()

np.float64(2468.8151197039497)

In [83]:

preg["agepreg"] = preg["agepreg"] / 100

In [84]:

preg["agepreg"].mean()

np.float64(24.6881511970395)

In [85]:

preg["birthwgt_oz"] = preg["birthwgt_oz"].replace([97, 98, 99], np.nan)

In [86]:

preg["totalwgt_lb"] = preg["birthwgt_lb"] + preg["birthwgt_oz"] / 16.0
preg["totalwgt_lb"].mean()

np.float64(7.265628457623368)

In [87]:

weights = preg["totalwgt_lb"]
n = weights.count()
n

np.int64(9038)

In [88]:
subset = preg.query("caseid == 10229")
subset.shape

(7, 244)

In [89]:
subset["outcome"].values

array([4, 4, 4, 4, 4, 4, 1])

In [90]:
preg["birthord"].value_counts(dropna=False).sort_index()

birthord
1.0     4413
2.0     2874
3.0     1234
4.0      421
5.0      126
6.0       50
7.0       20
8.0        7
9.0        2
10.0       1
NaN     4445
Name: count, dtype: int64

In [91]:
preg["birthwgt_kg"] = preg["totalwgt_lb"] * 0.45359237

In [92]:
preg["birthwgt_kg"].mean()

np.float64(3.295633631632828)

In [93]:
ans = preg.query("caseid ==2298")

In [94]:

ans["prglngth"].values

array([40, 36, 30, 40])

In [95]:

ans2 = preg.query("caseid == 5013 and pregordr == 1")


In [96]:
ans2.birthwgt_kg

5516    3.345244
Name: birthwgt_kg, dtype: float64